In [1]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom

import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)

import pyvista as pv

def remove_zeros(array):
    a = onp.array(array)
    a[onp.abs(a) < 1e-14] = 0.0
    return a

from jax_sn.domain.geometric_domain import GeometricDomain, TopologicalSortedAngle
from jax_sn.solution_domain.solution_domain import FEMMatrices, PrecomputedGeometricData
from jax_sn.domain.geometric_domain import CrossSectionData, CrossSectionDataSingleGroup
from jax_sn.source import Source
from jax_sn.operators.transport_operator import TransportOperatorBatched
from typing import List
from jax_sn.tools.mesh_tools import _pad_last_dimension
import matplotlib.pyplot as plt
from dataclasses import dataclass
onp.set_printoptions(linewidth = 200)
from finite_elements.test_moment_scatter_operator import _setup_all_sb

In [2]:
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.solution_domain import SolutionDomain
from jax_sn.domain import CrossSectionData, Domain

In [3]:
dimension = 3
degree = 2 
tn_order = 5
solution_domain, quad_set, toposorts, sweep_array, source = _setup_all_sb(dimension, degree, tn_order)

In [84]:
transport_operator = TransportOperatorBatched.from_solution_domain_data_and_toposort(solution_domain.solution_domain_data, toposorts, batch_size = 4)
moment_operator = jsn.operators.moment_operator.MomentOperator.from_quadrature_set(quad_set, 4)

In [85]:
inv_result = transport_operator.inv(source, 0)

In [108]:
from jax_sn.operators.transport_operator.transport_operator_forward import forward_transport_operator_multiple_angles_batch, forward_transport_operator_multiple_angles_vmap


result = forward_transport_operator_multiple_angles_batch(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), toposorts, inv_result, batch_size=4)

result2= forward_transport_operator_multiple_angles_vmap(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), toposorts, inv_result)

In [103]:
from jax_sn.operators.mass_operator.mass_operator_base import _mass_kernel_idx_in_precomp_vmap, _mass_kernel_idx_in_precomp_vmap_vmap_values
all_elements = jnp.arange(solution_domain.domain.n_elements)
print(source.source_values.shape)
source_mass_map = _mass_kernel_idx_in_precomp_vmap_vmap_values(all_elements, solution_domain.fem_matrices, solution_domain.precomputed_geometric_data, source.source_values)

(200, 378, 10)


In [104]:
onp.testing.assert_allclose(result, source_mass_map, rtol=1e-12)

In [105]:
source_mass_map[0]

Array([[-0.01052807, -0.01052807, -0.01052807, ...,  0.0421123 ,  0.0421123 ,  0.0421123 ],
       [-0.01077941, -0.01077941, -0.01077941, ...,  0.04311763,  0.04311763,  0.04311763],
       [-0.01103074, -0.01103074, -0.01103074, ...,  0.04412296,  0.04412296,  0.04412296],
       ...,
       [-0.03492602, -0.03492602, -0.03492602, ...,  0.1397041 ,  0.1397041 ,  0.1397041 ],
       [-0.0350098 , -0.0350098 , -0.0350098 , ...,  0.14003921,  0.14003921,  0.14003921],
       [-0.03509358, -0.03509358, -0.03509358, ...,  0.14037432,  0.14037432,  0.14037432]], dtype=float64)

In [106]:
def closure_node_functions()

SyntaxError: expected ':' (1786422932.py, line 1)